In [ ]:
# Part (a)import mathfrom statistics import NormalDistfrom IPython.display import SVG, displaydef load_series(path):    with open(path) as f:        return [float(x) for x in f.read().split()]def build_regression_matrices(series, order):    X = []    y = []    for t in range(order, len(series)):        row = [1.0]        for j in range(1, order + 1):            row.append(series[t - j])        X.append(row)        y.append(series[t])    return X, ydef solve_linear_system(A, b):    n = len(A)    aug = [A[i][:] + [b[i]] for i in range(n)]    for col in range(n):        pivot_row = max(range(col, n), key=lambda r: abs(aug[r][col]))        aug[col], aug[pivot_row] = aug[pivot_row], aug[col]        pivot = aug[col][col]        if abs(pivot) < 1e-12:            raise ValueError('Singular matrix encountered while solving normal equations.')        for j in range(col, n + 1):            aug[col][j] /= pivot        for r in range(n):            if r == col:                continue            factor = aug[r][col]            if factor == 0:                continue            for j in range(col, n + 1):                aug[r][j] -= factor * aug[col][j]    return [aug[i][n] for i in range(n)]def fit_ar(series, order):    X, y = build_regression_matrices(series, order)    p = len(X[0])    XtX = [[0.0] * p for _ in range(p)]    XtY = [0.0] * p    for row, target in zip(X, y):        for j in range(p):            XtY[j] += row[j] * target            for k in range(p):                XtX[j][k] += row[j] * row[k]    beta = solve_linear_system(XtX, XtY)    return beta, X, ydef compute_residuals(series, beta, order):    residuals = [None] * order    for t in range(order, len(series)):        pred = beta[0]        for j in range(1, order + 1):            pred += beta[j] * series[t - j]        residuals.append(series[t] - pred)    return residualsdef autocorrelation(data, lag):    m = len(data)    mean = sum(data) / m    denom = sum((x - mean) ** 2 for x in data)    num = sum((data[i] - mean) * (data[i - lag] - mean) for i in range(lag, m))    return num / denomdef save_acf_svg(acf_values, n_eff, filename, max_lag):    width, height = 800, 400    margin_left, margin_right, margin_top, margin_bottom = 60, 20, 40, 60    plot_width = width - margin_left - margin_right    plot_height = height - margin_top - margin_bottom    x_step = plot_width / max_lag    lb = -1.96 / math.sqrt(n_eff)    ub = 1.96 / math.sqrt(n_eff)    lines = []    lines.append(f"<svg width='{width}' height='{height}' xmlns='http://www.w3.org/2000/svg'>")    lines.append(f"<rect x='0' y='0' width='{width}' height='{height}' fill='white' />")    x0 = margin_left    y0 = margin_top + plot_height    lines.append(f"<line x1='{x0}' y1='{y0}' x2='{x0 + plot_width}' y2='{y0}' stroke='black' stroke-width='1' />")    lines.append(f"<line x1='{x0}' y1='{margin_top}' x2='{x0}' y2='{y0}' stroke='black' stroke-width='1' />")    zero_y = margin_top + plot_height * 0.5    lines.append(f"<line x1='{x0}' y1='{zero_y}' x2='{x0 + plot_width}' y2='{zero_y}' stroke='gray' stroke-dasharray='4 4' stroke-width='1' />")    for bound in (lb, ub):        y = margin_top + plot_height * (1 - (bound + 1) / 2)        lines.append(f"<line x1='{x0}' y1='{y}' x2='{x0 + plot_width}' y2='{y}' stroke='red' stroke-dasharray='4 4' stroke-width='1' />")    for lag in range(1, max_lag + 1):        value = acf_values[lag]        x = x0 + (lag - 0.5) * x_step        y = margin_top + plot_height * (1 - (value + 1) / 2)        lines.append(f"<line x1='{x}' y1='{zero_y}' x2='{x}' y2='{y}' stroke='steelblue' stroke-width='6' />")        lines.append(f"<text x='{x}' y='{y0 + 20}' font-size='12' text-anchor='middle'>{lag}</text>")    lines.append(f"<text x='{width / 2}' y='{25}' font-size='16' text-anchor='middle'>ACF of AR(3) Residuals</text>")    lines.append(f"<text x='{width / 2}' y='{height - 10}' font-size='14' text-anchor='middle'>Lag</text>")    lines.append(f"<text x='20' y='{margin_top + plot_height / 2}' font-size='14' text-anchor='middle' transform='rotate(-90 20 {margin_top + plot_height / 2})'>Autocorrelation</text>")    lines.append('</svg>')    with open(filename, 'w') as f:        f.write(''.join(lines))hare_series = load_series('hare.dat')order = 3beta, X_matrix, y_vector = fit_ar(hare_series, order)residuals = compute_residuals(hare_series, beta, order)filtered_residuals = [r for r in residuals if r is not None]max_lag = 20acf_values = [1.0] + [autocorrelation(filtered_residuals, lag) for lag in range(1, max_lag + 1)]save_acf_svg(acf_values, len(filtered_residuals), 'hare_residuals_acf.svg', max_lag)display(SVG(filename='hare_residuals_acf.svg'))

In [ ]:
# Part (b)import mathK = 9N = len(filtered_residuals)ljung_box_Q = N * (N + 2) * sum((acf_values[k] ** 2) / (N - k) for k in range(1, K + 1))print(f'Ljung-Box Q-statistic (K=9): {ljung_box_Q:.4f}')print(f'Degrees of freedom: {K - 3}')

In [ ]:
# Part (c)from statistics import NormalDistnonzero_residuals = [r for r in filtered_residuals if r != 0]signs = [1 if r > 0 else -1 for r in nonzero_residuals]runs = 1for i in range(1, len(signs)):    if signs[i] != signs[i - 1]:        runs += 1n_pos = sum(1 for s in signs if s == 1)n_neg = sum(1 for s in signs if s == -1)expected_runs = 1 + 2 * n_pos * n_neg / (n_pos + n_neg)variance_runs = (2 * n_pos * n_neg * (2 * n_pos * n_neg - n_pos - n_neg)) / (((n_pos + n_neg) ** 2) * (n_pos + n_neg - 1))z_runs = (runs - expected_runs) / math.sqrt(variance_runs)p_runs = 2 * (1 - NormalDist().cdf(abs(z_runs)))print(f'Runs count: {runs}')print(f'Expected runs: {expected_runs:.4f}')print(f'Z-statistic: {z_runs:.4f}')print(f'Two-sided p-value: {p_runs:.4f}')

In [ ]:
# Part (d)import mathfrom statistics import NormalDistfrom IPython.display import SVG, displayndist = NormalDist()sorted_residuals = sorted(filtered_residuals)N = len(sorted_residuals)qq_points = []for i, value in enumerate(sorted_residuals, start=1):    prob = (i - 0.375) / (N + 0.25)    theor = ndist.inv_cdf(prob)    qq_points.append((theor, value))sum_x = sum(pt[0] for pt in qq_points)sum_y = sum(pt[1] for pt in qq_points)sum_xx = sum(pt[0] ** 2 for pt in qq_points)sum_xy = sum(pt[0] * pt[1] for pt in qq_points)slope = (N * sum_xy - sum_x * sum_y) / (N * sum_xx - sum_x ** 2)intercept = (sum_y - slope * sum_x) / Nwidth, height = 800, 400margin = 60plot_width = width - 2 * marginplot_height = height - 2 * marginxs = [pt[0] for pt in qq_points]ys = [pt[1] for pt in qq_points]min_x, max_x = min(xs), max(xs)min_y, max_y = min(ys), max(ys)padding_x = 0.05 * (max_x - min_x)padding_y = 0.05 * (max_y - min_y)min_x -= padding_xmax_x += padding_xmin_y -= padding_ymax_y += padding_ydef to_svg_coords(x, y):    px = margin + (x - min_x) / (max_x - min_x) * plot_width    py = height - margin - (y - min_y) / (max_y - min_y) * plot_height    return px, pylines = []lines.append(f"<svg width='{width}' height='{height}' xmlns='http://www.w3.org/2000/svg'>")lines.append(f"<rect x='0' y='0' width='{width}' height='{height}' fill='white' />")lines.append(f"<line x1='{margin}' y1='{height - margin}' x2='{margin + plot_width}' y2='{height - margin}' stroke='black' stroke-width='1' />")lines.append(f"<line x1='{margin}' y1='{margin}' x2='{margin}' y2='{height - margin}' stroke='black' stroke-width='1' />")x1_line, y1_line = min_x, slope * min_x + interceptx2_line, y2_line = max_x, slope * max_x + interceptpx1, py1 = to_svg_coords(x1_line, y1_line)px2, py2 = to_svg_coords(x2_line, y2_line)lines.append(f"<line x1='{px1}' y1='{py1}' x2='{px2}' y2='{py2}' stroke='red' stroke-width='1.5' />")for x, y in qq_points:    px, py = to_svg_coords(x, y)    lines.append(f"<circle cx='{px}' cy='{py}' r='3' fill='steelblue' />")lines.append(f"<text x='{width / 2}' y='{25}' font-size='16' text-anchor='middle'>QQ Plot of AR(3) Residuals</text>")lines.append(f"<text x='{width / 2}' y='{height - 20}' font-size='14' text-anchor='middle'>Theoretical Quantiles</text>")lines.append(f"<text x='20' y='{height / 2}' font-size='14' text-anchor='middle' transform='rotate(-90 20 {height / 2})'>Sample Quantiles</text>")lines.append('</svg>')with open('hare_residuals_qq.svg', 'w') as f:    f.write(''.join(lines))display(SVG(filename='hare_residuals_qq.svg'))

In [ ]:
# Part (e)import mathimport randomfrom statistics import NormalDistndist = NormalDist()N = len(filtered_residuals)sorted_residuals = sorted(filtered_residuals)coefficients = [ndist.inv_cdf((i - 0.375) / (N + 0.25)) for i in range(1, N + 1)]norm_factor = math.sqrt(sum(c * c for c in coefficients))a_weights = [c / norm_factor for c in coefficients]mean_residual = sum(filtered_residuals) / Ndenom = sum((r - mean_residual) ** 2 for r in filtered_residuals)numer = sum(a * x for a, x in zip(a_weights, sorted_residuals)) ** 2W_statistic = numer / denomrandom.seed(2024)iterations = 5000count = 0for _ in range(iterations):    sample = sorted(random.gauss(0, 1) for _ in range(N))    mean_sample = sum(sample) / N    denom_sample = sum((s - mean_sample) ** 2 for s in sample)    numer_sample = sum(a * s for a, s in zip(a_weights, sample)) ** 2    if denom_sample == 0:        continue    W_sample = numer_sample / denom_sample    if W_sample <= W_statistic:        count += 1p_value = count / iterationsprint(f'Shapiro-Wilk W statistic (Monte Carlo approximation): {W_statistic:.4f}')print(f'Approximate p-value: {p_value:.4f}')